# Empirical Baseline Uplift using simple statistics for Model 3

This script calculates the Empirical Baseline Uplift using simple statistics. It proves to the that the relational logic works, verifies we have enough households in the Control group, and gives a real number 

In [3]:
import pandas as pd
import os

def analyze_causal_feasibility(campaign_id):
    """
    Diagnostic script for Model 3. Checks Control Group volume risk, executes
    the Day-to-Week time translation, and calculates baseline causal uplift 
    by controlling for in-store displays using causal_data.
    """
    # 1. Exact Paths based on your directory
    data_dir = '/Users/shreyashetty/Documents/Projects_and_Assignments/DSI_Capstone/Agentic-AI-Marketing-Decision-Lab/data/dunnhumby_The-Complete-Journey CSV/'
    
    print(f"--- Model 3 Causal Feasibility & Risk Check: Campaign {campaign_id} ---")
    
    # 2. Load specific columns to save memory and optimize processing
    tx = pd.read_csv(os.path.join(data_dir, 'transaction_data.csv'), usecols=['household_key', 'PRODUCT_ID', 'WEEK_NO', 'STORE_ID'])
    camp_desc = pd.read_csv(os.path.join(data_dir, 'campaign_desc.csv'))
    camp_table = pd.read_csv(os.path.join(data_dir, 'campaign_table.csv'))
    coupons = pd.read_csv(os.path.join(data_dir, 'coupon.csv'))
    causal = pd.read_csv(os.path.join(data_dir, 'causal_data.csv'))
    
    # Extract unique base households from transactions (more reliable than demographics)
    all_hhs = set(tx['household_key'].unique())
    
    # 3. Map Campaign Parameters & Time Translation (Mitigating Risk 1)
    camp_info = camp_desc[camp_desc['CAMPAIGN'] == campaign_id].iloc[0]
    target_products = coupons[coupons['CAMPAIGN'] == campaign_id]['PRODUCT_ID'].unique()
    
    # Translating Days to Weeks to align with causal_data
    start_week = camp_info['START_DAY'] // 7
    end_week = camp_info['END_DAY'] // 7
    
    # 4. Define Treatment & Control (Mitigating Risk 2: Volume Check)
    treated_hhs = set(camp_table[camp_table['CAMPAIGN'] == campaign_id]['household_key'].unique())
    control_hhs = all_hhs - treated_hhs
    
    print(f"\n[Risk Diagnostic 1: Control Group Volume Check]")
    print(f"Treatment Group: {len(treated_hhs)} households")
    print(f"Control Group: {len(control_hhs)} households")
    if len(control_hhs) < 200:
        print("WARNING: Control Group is too small for statistical significance. ML Training will fail.")
    else:
        print("STATUS: Control group volume is healthy.")
        
    # 5. Filter Transactions to Campaign Window and Target Products
    window_tx = tx[(tx['WEEK_NO'] >= start_week) & (tx['WEEK_NO'] <= end_week)]
    target_tx = window_tx[window_tx['PRODUCT_ID'].isin(target_products)]
    
    # 6. Join Causal Data to Identify Confounders (Displays / Circulars)
    merged_tx = pd.merge(target_tx, causal, on=['PRODUCT_ID', 'STORE_ID', 'WEEK_NO'], how='left')
    
    # '0' means no display/mailer. Flag anything that is NOT '0' and NOT null.
    merged_tx['is_confounded'] = (
        ((merged_tx['display'].notna()) & (merged_tx['display'].astype(str) != '0')) | 
        ((merged_tx['mailer'].notna()) & (merged_tx['mailer'].astype(str) != '0'))
    ).astype(int)
    
    print(f"\n[Risk Diagnostic 2: Confounder Tracking via causal_data]")
    print(f"Total purchases of target products during campaign: {len(merged_tx)}")
    print(f"Purchases driven by in-store displays/mass circulars (Confounded): {merged_tx['is_confounded'].sum()}")
    
    # 7. Calculate True Empirical Uplift (Strictly unconfounded transactions)
    pure_tx = merged_tx[merged_tx['is_confounded'] == 0]
    
    purchasing_treated = set(pure_tx['household_key'].unique()).intersection(treated_hhs)
    purchasing_control = set(pure_tx['household_key'].unique()).intersection(control_hhs)
    
    treatment_rate = len(purchasing_treated) / len(treated_hhs) if treated_hhs else 0
    control_rate = len(purchasing_control) / len(control_hhs) if control_hhs else 0
    
    print(f"\n[Diagnostic 3: Baseline Empirical Uplift]")
    print(f"Treatment Conversion Rate (Unconfounded): {treatment_rate * 100:.2f}%")
    print(f"Control Baseline Conversion Rate (Unconfounded): {control_rate * 100:.2f}%")
    print(f"--> True Incremental Uplift Baseline: {(treatment_rate - control_rate) * 100:.2f}%")

# Execute to prove feasibility
analyze_causal_feasibility(campaign_id=8)

--- Model 3 Causal Feasibility & Risk Check: Campaign 8 ---

[Risk Diagnostic 1: Control Group Volume Check]
Treatment Group: 1076 households
Control Group: 1424 households
STATUS: Control group volume is healthy.

[Risk Diagnostic 2: Confounder Tracking via causal_data]
Total purchases of target products during campaign: 84544
Purchases driven by in-store displays/mass circulars (Confounded): 25718

[Diagnostic 3: Baseline Empirical Uplift]
Treatment Conversion Rate (Unconfounded): 96.56%
Control Baseline Conversion Rate (Unconfounded): 72.33%
--> True Incremental Uplift Baseline: 24.23%
